# ChagaSight Training - Fold 0 (COMPLETE WORKING VERSION)

**ALL BUGS FIXED:**
- ✅ vit_1d_fm.py contiguity issue fixed
- ✅ Dataset custom collate function
- ✅ Checkpoint resumption
- ✅ Official PhysioNet metrics

**IMPORTANT**: Batch size set to 16 (not 32) for 6GB GPU

In [1]:
# Cell 1: Imports and Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders
from src.training.trainer import ChagasTrainer

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {mem_gb:.2f} GB")
    if mem_gb < 8:
        print("⚠️  GPU has <8GB memory. Using batch_size=16 (not 32)")
else:
    print("⚠️  WARNING: No GPU detected. Training will be VERY slow on CPU.")

print("\n✓ All imports successful!")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
Memory: 6.44 GB
⚠️  GPU has <8GB memory. Using batch_size=16 (not 32)

✓ All imports successful!


In [2]:
# Cell 2: Configuration
FOLD = 0  # Change to 1, 2, 3, 4 for other notebooks

# Paths
DATA_DIR = project_root / 'data' / 'processed'
METADATA_CSV = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR = DATA_DIR / '2d_images'
SIGNALS_DIR = DATA_DIR / '1d_signals_100hz'

# Verify files exist
if not METADATA_CSV.exists():
    raise FileNotFoundError(f"Metadata CSV not found: {METADATA_CSV}")
print(f"✓ Metadata CSV: {METADATA_CSV}")

# Checkpoints
CHECKPOINT_DIR = project_root / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Pretrained weights (optional)
MAE_CHECKPOINT = CHECKPOINT_DIR / 'mae_2d_pretrained.pt'
STMEM_CHECKPOINT = CHECKPOINT_DIR / 'stmem_1d_pretrained.pt'

# Training config
# IMPORTANT: Use batch_size=16 for 6GB GPU (32 causes OOM)
BATCH_SIZE = 16
NUM_WORKERS = 4
USE_AMP = True

# Phase 1 (FM frozen)
PHASE1_ITERATIONS = 2000
PHASE1_LR = 2e-4

# Phase 2 (FM unfrozen)
PHASE2_ITERATIONS = 12000
PHASE2_LR_HIGH = 2e-4
PHASE2_LR_LOW = 2e-5

# Checkpoint resumption
RESUME_FROM = None  # Set to resume
# Example: RESUME_FROM = str(CHECKPOINT_DIR / 'fold0_latest.pt')

print(f"\n✓ Configuration for Fold {FOLD}:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Phase 1: {PHASE1_ITERATIONS} iterations")
print(f"  Phase 2: {PHASE2_ITERATIONS} iterations")
print(f"  Total: {PHASE1_ITERATIONS + PHASE2_ITERATIONS} iterations")

✓ Metadata CSV: d:\IIT\L6\FYP\ChagaSight\data\processed\metadata\combined_5fold.csv

✓ Configuration for Fold 0:
  Batch size: 16
  Phase 1: 2000 iterations
  Phase 2: 12000 iterations
  Total: 14000 iterations


In [3]:
# Cell 3: Create Dataloaders
print("Creating dataloaders...")

train_loader, val_loader = create_dataloaders(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    fold=FOLD,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampling=True,
    augment_train=True
)

# Test batch
print("\nTesting batch...")
batch = next(iter(train_loader))
print(f"✓ Batch shapes:")
print(f"  image:  {batch['image'].shape}")  # (16, 3, 24, 2048)
print(f"  signal: {batch['signal'].shape}")  # (16, 12, 1000)
print(f"  age:    {batch['age'].shape}")  # (16,)
print(f"  sex:    {batch['sex'].shape}")  # (16,)
print(f"  label:  {batch['label'].shape}")  # (16,)
print(f"\n✓ Dataloaders ready!")

Creating dataloaders...


d:\IIT\L6\FYP\ChagaSight\src\training\dataset.py:89: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(metadata_csv)


✓ Loaded train fold 0: 66504 samples
  Datasets: {'ptbxl': 17439, 'samitrop': 1305, 'code15': 47760}
  Positive: 2277, Negative: 64227
✓ Loaded val fold 0: 16626 samples
  Datasets: {'ptbxl': 4360, 'samitrop': 326, 'code15': 11940}
  Positive: 569, Negative: 16057

✓ Created dataloaders for fold 0:
  Train: 66504 samples, 4156 batches
  Val:   16626 samples, 1040 batches
  Weighted sampling: True
  Augmentation: True

Testing batch...


d:\IIT\L6\FYP\ChagaSight\src\training\dataset.py:89: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(metadata_csv)


✓ Batch shapes:
  image:  torch.Size([16, 3, 24, 2048])
  signal: torch.Size([16, 12, 1000])
  age:    torch.Size([16])
  sex:    torch.Size([16])
  label:  torch.Size([16])

✓ Dataloaders ready!


In [4]:
# Cell 4: Create Model
print("Creating model...")

model = HybridChagasModel(
    img_size=(24, 2048),
    patch_size_2d=(8, 64),
    num_leads=12,
    seq_len_1d=1000,
    patch_size_1d=50,
    embed_dim=768,
    depth=12,
    num_heads=12,
    use_aol=True,
    use_demographics=True
)

# Load pretrained weights
if MAE_CHECKPOINT.exists():
    print(f"✓ Loading MAE weights from {MAE_CHECKPOINT}")
    model.vit_2d.load_mae_pretrained(str(MAE_CHECKPOINT))
else:
    print(f"⚠️  No MAE checkpoint. Training 2D-ViT from scratch.")

if STMEM_CHECKPOINT.exists():
    print(f"✓ Loading ST-MEM weights from {STMEM_CHECKPOINT}")
    model.vit_1d_fm.load_stmem_pretrained(str(STMEM_CHECKPOINT))
else:
    print(f"⚠️  No ST-MEM checkpoint. Training 1D-ViT FM from scratch.")

model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n✓ Model ready:")
print(f"  Total params:     {total_params:,}")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Device: {device}")

Creating model...
⚠️  No MAE checkpoint. Training 2D-ViT from scratch.
⚠️  No ST-MEM checkpoint. Training 1D-ViT FM from scratch.

✓ Model ready:
  Total params:     173,570,817
  Trainable params: 173,570,817
  Device: cuda


In [5]:
# Cell 5: Create Trainer
print("Creating trainer...")

trainer = ChagasTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    phase1_iterations=PHASE1_ITERATIONS,
    phase2_iterations=PHASE2_ITERATIONS,
    phase1_lr=PHASE1_LR,
    phase2_lr_high=PHASE2_LR_HIGH,
    phase2_lr_low=PHASE2_LR_LOW,
    checkpoint_dir=str(CHECKPOINT_DIR),
    use_amp=USE_AMP,
    val_every_n_iters=500
)

print("✓ Trainer created")
print("\n" + "="*60)
print("READY TO TRAIN!")
print("="*60)
print("Expected time:")
print("  Phase 1: ~45-60 minutes")
print("  Phase 2: ~4-5 hours")
print("  Total:   ~5-6 hours")
print("="*60)

Creating trainer...
✓ Trainer created

READY TO TRAIN!
Expected time:
  Phase 1: ~45-60 minutes
  Phase 2: ~4-5 hours
  Total:   ~5-6 hours


d:\IIT\L6\FYP\ChagaSight\src\training\trainer.py:60: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if use_amp else None


In [6]:
# Cell 6: TRAIN!
print("\nStarting training...\n")

metrics = trainer.train(fold=FOLD, resume_from=RESUME_FROM)

# Display results
print(f"\n" + "="*70)
print(f" Final Results - Fold {FOLD}")
print("="*70)
print(f"TPR@5%:  {metrics['tpr_5pct']:.4f}  ⭐ PRIMARY METRIC")
print(f"AUROC:   {metrics['auroc']:.4f}")
print(f"AUPRC:   {metrics['auprc']:.4f}")
print("="*70)

# Target check
if metrics['tpr_5pct'] >= 0.42:
    print("✅ TARGET ACHIEVED! (≥0.42)")
elif metrics['tpr_5pct'] >= 0.35:
    print("⚠️  Good progress, but below target 0.42")
else:
    print("❌ Below minimum threshold 0.35")

# Compare to top team
top_team = 0.445
if metrics['tpr_5pct'] >= top_team:
    print(f"🎉 EXCELLENT! Matches/beats top team ({top_team})")
else:
    gap = top_team - metrics['tpr_5pct']
    print(f"Gap to top team: {gap:.4f}")

print("\n✓ Training complete!")
print(f"Best model saved: {CHECKPOINT_DIR / f'fold{FOLD}_best.pt'}")


Starting training...


Training Fold 0

📌 PHASE 1: FM Frozen (2000 iterations)
✓ FM frozen


Phase 1:   0%|          | 0/2000 [00:00<?, ?it/s]d:\IIT\L6\FYP\ChagaSight\src\training\trainer.py:198: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.use_amp):
Phase 1:  25%|██▌       | 500/2000 [02:47<08:00,  3.12it/s, loss=inf] d:\IIT\L6\FYP\ChagaSight\src\training\trainer.py:234: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.use_amp):


KeyboardInterrupt: 

In [ ]:
# Cell 7: Save Results
results_df = pd.DataFrame([metrics])
results_df['fold'] = FOLD
results_csv = CHECKPOINT_DIR / f"fold{FOLD}_results.csv"
results_df.to_csv(results_csv, index=False)

print(f"✓ Results saved to {results_csv}")
print(f"\nResults:")
print(results_df.to_string(index=False))

print(f"\n" + "="*70)
print("NEXT STEPS:")
print("="*70)
print("1. Use evaluation_complete.ipynb for detailed analysis")
print("2. Train remaining folds (1, 2, 3, 4)")
print("3. Compute ensemble metrics across all folds")
print("="*70)